# Aperture-Limited Gaussian Beam

Approximate a circular aperture with a packet of Gaussian beams, propagate it to a detector, and compare the input aperture field with the propagated diffraction pattern. This notebook intentionally uses public TemGym APIs only; CUDA/Pallas kernel details live in `temgym_core.evaluate`.


In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import jax

from temgym_core.components import Detector
from temgym_core.evaluate import evaluate_gaussians_jax_scan
from temgym_core.run import run_to_end
from temgym_core.source import circular_input_wave

jax.config.update("jax_enable_x64", True)


## Input Aperture

The Gaussian packet samples a circular aperture. The overlap factor controls how densely neighbouring beamlets cover the aperture.


In [ ]:
voltage = 200e3
aperture_radius = 1.0e-6
waist = 120e-9
window = 4.0e-6
shape = (160, 160)
pixel = window / shape[0]

input_grid = Detector(z=0.0, pixel_size=(pixel, pixel), shape=shape)
detector = Detector(z=1.0e-3, pixel_size=(pixel, pixel), shape=shape)

rays_in = circular_input_wave(
    aperture_radius=aperture_radius,
    waist=waist,
    voltage=voltage,
    overlap_factor=2.0,
).to_vector()
print(f"Gaussian beamlets: {np.asarray(rays_in.x).size}")


## Propagate and Evaluate

The same evaluator is used at the input and detector planes. For larger apertures, switch to the GPU wrapper if CUDA-enabled JAX is available.


In [ ]:
input_field = evaluate_gaussians_jax_scan(rays_in, input_grid, batch_size=128)
rays_out = run_to_end(rays_in, (detector,))
output_field = evaluate_gaussians_jax_scan(rays_out, detector, batch_size=128)

input_field = np.asarray(input_field)
output_field = np.asarray(output_field)


## Visualise

The input field shows the filled aperture; the output field shows the far-field diffraction structure produced by the finite aperture.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

im0 = ax[0].imshow(np.abs(input_field) ** 2, extent=input_grid.extent, origin="lower", cmap="inferno")
ax[0].set_title("Input aperture intensity")
ax[0].set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(np.abs(output_field) ** 2, extent=detector.extent, origin="lower", cmap="inferno")
ax[1].set_title("Propagated intensity")
ax[1].set_xlabel("x (m)")
ax[1].set_ylabel("y (m)")
fig.colorbar(im1, ax=ax[1])
